# Chapter 16: Machine Learning Capstone Project — Solutions

## End-to-End Classification on the Wine Dataset

This notebook contains complete solutions with analysis and insights for the
capstone project. The Wine dataset is a classic multiclass classification problem
with 13 numerical features describing chemical properties of wines from 3 cultivars.

We will walk through the full ML workflow: exploration, preprocessing, model
comparison, hyperparameter tuning, and final evaluation.

---
## Setup: Imports and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('Setup complete.')

---
## Part 1: Data Exploration

### Q1: Load the Wine Dataset

Load the wine dataset using `load_wine()`. Create a pandas DataFrame from the
feature data with proper column names, and add a `target` column for the class labels.

Display:
- The shape of the DataFrame
- The first 5 rows
- The target names (class labels)

In [ ]:
# Load the wine dataset
wine = load_wine()

# Create a DataFrame with feature names as columns
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

print(f'Dataset shape: {df.shape}')
print(f'Number of features: {len(wine.feature_names)}')
print(f'Number of classes: {len(wine.target_names)}')
print(f'Target names: {list(wine.target_names)}')
print()
df.head()

**Insight:** The Wine dataset contains 178 samples with 13 numerical features each.
The three classes represent different wine cultivars (class_0, class_1, class_2).
Features capture chemical properties like alcohol content, malic acid, ash,
alkalinity, magnesium, phenols, flavanoids, color intensity, hue, OD280/OD315,
and proline. All features are continuous, making this a clean dataset for
classification experiments.

### Q2: Summary Statistics and Missing Values

Display the summary statistics for all features using `.describe()`.
Then check whether there are any missing values in the dataset.

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Data types
print('Data types:')
print(df.dtypes)

**Insight:** There are no missing values in the dataset, which is expected for
sklearn's built-in datasets. The features vary significantly in scale: for example,
proline ranges from ~278 to ~1680, while the OD280/OD315 ratio ranges from ~1.27
to ~4.0. This large difference in feature scales means that standardization will
be critical for distance-based models like SVM and for PCA.

### Q3: Class Distribution

Show the distribution of the target classes. Are they balanced?
Create a bar plot showing the count of samples per class.

In [ ]:
# Class distribution
class_counts = df['target'].value_counts().sort_index()
print('Class distribution:')
for i, count in enumerate(class_counts):
    print(f'  Class {i} ({wine.target_names[i]}): {count} samples ({count/len(df)*100:.1f}%)')

In [ ]:
# Bar plot of class distribution
fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette('Set2', n_colors=3)
bars = ax.bar(wine.target_names, class_counts.values, color=colors, edgecolor='black')

# Add count labels on bars
for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(count), ha='center', va='bottom', fontweight='bold')

ax.set_xlabel('Wine Class')
ax.set_ylabel('Number of Samples')
ax.set_title('Class Distribution in the Wine Dataset')
plt.tight_layout()
plt.show()

**Insight:** The classes are reasonably balanced but not perfectly so. Class 1
(the largest) has 71 samples while class 2 (the smallest) has 48 samples. The
ratio between the largest and smallest class is about 1.5:1, which is mild
imbalance. For this level of imbalance, stratified splitting and stratified
cross-validation (which we will use) should be sufficient — we do not need to
resort to oversampling or class weights.

### Q4: Correlation Heatmap

Create a correlation heatmap of all 13 features.
Which pairs of features show the strongest positive and negative correlations?

In [ ]:
# Compute correlation matrix (features only, excluding target)
corr_matrix = df.drop(columns='target').corr()

# Create heatmap
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Find the strongest correlations (excluding diagonal)
corr_pairs = corr_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) != corr_pairs.index.get_level_values(1)]
corr_pairs = corr_pairs.drop_duplicates()

print('Top 5 strongest positive correlations:')
top_pos = corr_pairs.nlargest(5)
for (f1, f2), val in top_pos.items():
    print(f'  {f1} <-> {f2}: {val:.3f}')

print('\nTop 5 strongest negative correlations:')
top_neg = corr_pairs.nsmallest(5)
for (f1, f2), val in top_neg.items():
    print(f'  {f1} <-> {f2}: {val:.3f}')

**Insight:** The strongest positive correlation is between total_phenols and
flavanoids (~0.86), which makes chemical sense since flavanoids are a subclass
of phenolic compounds. The strongest negative correlation is between hue and
malic_acid (~-0.56). Several feature pairs show moderate correlations, suggesting
some redundancy in the feature space — this is why PCA can be effective at
reducing dimensionality while preserving information.

---
## Part 2: Preprocessing & Feature Engineering

### Q5: Train/Test Split

Split the data into training (80%) and test (20%) sets.
Use `stratify` to preserve the class distribution and `random_state=42` for reproducibility.

Print the shapes of X_train, X_test, y_train, y_test.

In [ ]:
# Separate features and target
X = df.drop(columns='target').values
y = df['target'].values

# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape:  {y_test.shape}')

In [ ]:
# Verify stratification preserved class proportions
print('\nClass proportions:')
for i in range(3):
    train_prop = (y_train == i).sum() / len(y_train) * 100
    test_prop = (y_test == i).sum() / len(y_test) * 100
    print(f'  Class {i}: Train={train_prop:.1f}%, Test={test_prop:.1f}%')

**Insight:** The stratified split gives us 142 training samples and 36 test samples.
The class proportions are well-preserved between train and test sets, which is
important for getting a reliable estimate of model performance on unseen data.

### Q6: Preprocessing Pipeline with StandardScaler

Build a preprocessing pipeline that applies `StandardScaler` to the features.
Fit the scaler on the training data and transform both training and test sets.

Verify the scaling by printing the mean and standard deviation of a few features
before and after scaling.

In [ ]:
# Create and fit the scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Verify scaling: compare before and after for first 3 features
feature_names = wine.feature_names
print('Before scaling (training set):')
for i in range(3):
    print(f'  {feature_names[i]:>25s}: mean={X_train[:, i].mean():.3f}, std={X_train[:, i].std():.3f}')

print('\nAfter scaling (training set):')
for i in range(3):
    print(f'  {feature_names[i]:>25s}: mean={X_train_scaled[:, i].mean():.3f}, std={X_train_scaled[:, i].std():.3f}')

**Insight:** After scaling, each feature in the training set has a mean of
approximately 0 and standard deviation of approximately 1. This is critical for
algorithms like SVM and Logistic Regression that are sensitive to feature scales.
Note that we fit the scaler on the training data only and then transform the test
data — this prevents data leakage from the test set.

### Q7: PCA — Dimensionality Reduction

Apply PCA to the scaled training data. Determine how many components are needed
to explain at least **95% of the total variance**.

Create a plot showing the cumulative explained variance ratio vs. number of components.

In [ ]:
# Apply PCA with all components to analyze variance
pca_full = PCA(random_state=42)
pca_full.fit(X_train_scaled)

# Cumulative explained variance
cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

# Find number of components for 95% variance
n_components_95 = np.argmax(cumulative_var >= 0.95) + 1
print(f'Number of components to explain 95% variance: {n_components_95}')
print(f'Variance explained by {n_components_95} components: {cumulative_var[n_components_95 - 1]:.4f}')

In [ ]:
# Plot cumulative explained variance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Individual explained variance
ax1.bar(range(1, 14), pca_full.explained_variance_ratio_, color='steelblue', edgecolor='black')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance Ratio')
ax1.set_title('Individual Explained Variance per Component')
ax1.set_xticks(range(1, 14))

# Cumulative explained variance
ax2.plot(range(1, 14), cumulative_var, 'bo-', linewidth=2, markersize=8)
ax2.axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
ax2.axvline(x=n_components_95, color='g', linestyle='--', alpha=0.7,
            label=f'{n_components_95} components')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Explained Variance')
ax2.set_title('Cumulative Explained Variance Ratio')
ax2.set_xticks(range(1, 14))
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Individual variance contributions
print('Explained variance per component:')
for i, var in enumerate(pca_full.explained_variance_ratio_):
    marker = ' <-- 95% reached' if i + 1 == n_components_95 else ''
    print(f'  PC{i+1:2d}: {var:.4f} (cumulative: {cumulative_var[i]:.4f}){marker}')

**Insight:** Approximately 8-10 principal components are needed to explain 95%
of the variance (the exact number depends on the random split). The first two
components alone capture a substantial portion of the variance. While PCA can
reduce dimensionality from 13 to fewer features, the reduction is moderate here.
For the model comparison that follows, we will use the original 13 scaled features
rather than PCA-transformed features, since the dataset is small enough that
dimensionality is not a bottleneck.

---
## Part 3: Model Training & Comparison

### Q8: Logistic Regression

Train a Logistic Regression model on the **scaled** training data.
Use `max_iter=5000` and `random_state=42`.

Report:
- Training accuracy
- Test accuracy
- Classification report on the test set

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=5000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Predictions
lr_train_pred = lr_model.predict(X_train_scaled)
lr_test_pred = lr_model.predict(X_test_scaled)

# Accuracies
lr_train_acc = accuracy_score(y_train, lr_train_pred)
lr_test_acc = accuracy_score(y_test, lr_test_pred)

print(f'Logistic Regression Results:')
print(f'  Training accuracy: {lr_train_acc:.4f}')
print(f'  Test accuracy:     {lr_test_acc:.4f}')

In [ ]:
# Classification report
print('Classification Report (Test Set):')
print(classification_report(y_test, lr_test_pred, target_names=wine.target_names))

**Insight:** Logistic Regression performs very well on the Wine dataset, achieving
high accuracy on both training and test sets. The model generalizes well with
minimal overfitting, which suggests the decision boundaries between the wine
classes are relatively linear in the scaled feature space. The classification
report shows strong precision and recall across all three classes.

### Q9: Random Forest

Train a Random Forest classifier with `n_estimators=100` and `random_state=42`
on the scaled training data.

Report:
- Training accuracy
- Test accuracy
- A bar plot of feature importances (top 10)

In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Predictions
rf_train_pred = rf_model.predict(X_train_scaled)
rf_test_pred = rf_model.predict(X_test_scaled)

# Accuracies
rf_train_acc = accuracy_score(y_train, rf_train_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

print(f'Random Forest Results:')
print(f'  Training accuracy: {rf_train_acc:.4f}')
print(f'  Test accuracy:     {rf_test_acc:.4f}')

In [ ]:
# Feature importances
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=wine.feature_names).sort_values(ascending=True)

# Plot top 10 feature importances
fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.tail(10).plot(kind='barh', color='forestgreen', edgecolor='black', ax=ax)
ax.set_xlabel('Feature Importance')
ax.set_title('Random Forest — Top 10 Feature Importances')
plt.tight_layout()
plt.show()

In [ ]:
# Print all feature importances sorted
print('Feature importances (sorted):')
for feat, imp in feat_imp.iloc[::-1].items():
    print(f'  {feat:>30s}: {imp:.4f}')

**Insight:** Random Forest identifies alcohol, proline, flavanoids, color_intensity,
and OD280/OD315_of_diluted_wines as the most discriminative features for
classifying wine varieties. The model achieves perfect training accuracy (1.0),
which is typical for Random Forests — the ensemble of deep trees can memorize the
training set. The gap between training and test accuracy indicates some overfitting,
though test performance is still strong.

### Q10: Support Vector Machine (SVM)

Train an SVM with an RBF kernel on the scaled training data.
Use `random_state=42`.

Report:
- Training accuracy
- Test accuracy

In [ ]:
# Train SVM with RBF kernel
svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train_scaled, y_train)

# Predictions
svm_train_pred = svm_model.predict(X_train_scaled)
svm_test_pred = svm_model.predict(X_test_scaled)

# Accuracies
svm_train_acc = accuracy_score(y_train, svm_train_pred)
svm_test_acc = accuracy_score(y_test, svm_test_pred)

print(f'SVM (RBF kernel) Results:')
print(f'  Training accuracy: {svm_train_acc:.4f}')
print(f'  Test accuracy:     {svm_test_acc:.4f}')

**Insight:** SVM with an RBF kernel performs competitively on this dataset.
Because we scaled the features beforehand, SVM can effectively measure distances
in the feature space. The RBF kernel maps the data to a higher-dimensional space
where nonlinear decision boundaries become possible, though for this dataset the
boundaries appear to be nearly linear.

### Q11: Model Comparison with Cross-Validation

Compare all 3 models using 5-fold stratified cross-validation on the training set.

Create a **box plot** showing the distribution of cross-validation accuracy scores
for each model. Which model performs best on average?

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42),
}

# Cross-validation scores
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f'{name:>22s}: mean={scores.mean():.4f}, std={scores.std():.4f}, scores={np.round(scores, 4)}')

In [ ]:
# Box plot comparison
fig, ax = plt.subplots(figsize=(10, 6))

labels = list(cv_results.keys())
data = [cv_results[name] for name in labels]

bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.5)

colors = ['#4C72B0', '#55A868', '#C44E52']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Accuracy')
ax.set_title('5-Fold Cross-Validation: Model Comparison')
ax.set_ylim(0.85, 1.02)

# Add mean markers
means = [np.mean(d) for d in data]
ax.scatter(range(1, len(means) + 1), means, color='gold', marker='D', s=80,
           zorder=3, label='Mean')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Identify the best model
best_model_name = max(cv_results, key=lambda k: cv_results[k].mean())
print(f'\nBest model by mean CV accuracy: {best_model_name} '
      f'({cv_results[best_model_name].mean():.4f})')

**Insight:** All three models perform well on the Wine dataset, typically achieving
95%+ accuracy in cross-validation. Logistic Regression and SVM tend to be the
top performers, benefiting from the feature scaling. Random Forest also does well
but may show slightly more variance across folds. The box plot visualizes not just
the mean performance but also the stability (spread) of each model's scores across
folds — a model with both high mean and low variance is preferred.

---
## Part 4: Hyperparameter Tuning

### Q12: GridSearchCV on the Best Model

Based on the cross-validation results from Q11, select the best-performing model.
Define a reasonable hyperparameter grid and run `GridSearchCV` with 5-fold
stratified cross-validation.

In [ ]:
# We will tune Logistic Regression as it typically performs best.
# If SVM was your best model, adjust the grid accordingly.

# Hyperparameter grid for Logistic Regression
lr_param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear', 'newton-cg'],
    'penalty': ['l2'],
}

lr_grid_search = GridSearchCV(
    LogisticRegression(max_iter=5000, random_state=42),
    param_grid=lr_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)

lr_grid_search.fit(X_train_scaled, y_train)
print(f'\nGrid search complete.')

In [ ]:
# Also tune SVM for comparison
svm_param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf'],
}

svm_grid_search = GridSearchCV(
    SVC(random_state=42),
    param_grid=svm_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
)

svm_grid_search.fit(X_train_scaled, y_train)
print(f'\nGrid search complete.')

In [ ]:
# Compare grid search results
print(f'Logistic Regression best CV score: {lr_grid_search.best_score_:.4f}')
print(f'SVM best CV score:                 {svm_grid_search.best_score_:.4f}')

# Select the overall best
if lr_grid_search.best_score_ >= svm_grid_search.best_score_:
    best_grid = lr_grid_search
    best_name = 'Logistic Regression'
else:
    best_grid = svm_grid_search
    best_name = 'SVM (RBF)'

print(f'\nSelected model for final evaluation: {best_name}')

**Insight:** GridSearchCV systematically evaluates every combination of
hyperparameters using 5-fold cross-validation. By tuning both Logistic Regression
and SVM, we ensure we are selecting not just the best algorithm but also its best
configuration. The regularization parameter C controls the trade-off between
fitting the training data and keeping the model simple — too small and the model
underfits, too large and it overfits.

### Q13: Best Parameters and Final Test Accuracy

Print the best hyperparameters found by GridSearchCV.
Evaluate the best estimator on the held-out test set and report the final accuracy.

In [ ]:
# Best parameters
print(f'Best model: {best_name}')
print(f'Best parameters: {best_grid.best_params_}')
print(f'Best CV accuracy: {best_grid.best_score_:.4f}')

# Evaluate on test set
best_estimator = best_grid.best_estimator_
final_test_pred = best_estimator.predict(X_test_scaled)
final_test_acc = accuracy_score(y_test, final_test_pred)

print(f'\nFinal test accuracy: {final_test_acc:.4f}')

**Insight:** The tuned model achieves strong accuracy on the held-out test set.
The close agreement between cross-validation accuracy and test accuracy indicates
that our model selection process was sound and we are not overfitting to the
validation folds. The hyperparameter tuning may yield only a modest improvement
over the defaults because the Wine dataset is relatively clean and well-separated.

### Q14: Final Confusion Matrix and Classification Report

Using the best tuned model from Q13:
1. Generate predictions on the test set
2. Display the confusion matrix as a heatmap
3. Print the full classification report

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, final_test_pred)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=wine.target_names)
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title(f'Confusion Matrix — {best_name} (Tuned)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Full classification report
print(f'Final Classification Report ({best_name}, tuned):')
print('=' * 60)
print(classification_report(y_test, final_test_pred, target_names=wine.target_names))

In [ ]:
# Per-class analysis
print('Per-class analysis:')
for i, name in enumerate(wine.target_names):
    correct = cm[i, i]
    total = cm[i].sum()
    print(f'  {name}: {correct}/{total} correct ({correct/total*100:.1f}%)')
    misclassified = [(wine.target_names[j], cm[i, j]) for j in range(3) if j != i and cm[i, j] > 0]
    if misclassified:
        for wrong_name, count in misclassified:
            print(f'    -> misclassified as {wrong_name}: {count}')
    else:
        print(f'    -> no misclassifications')

**Insight:** The confusion matrix reveals which wine classes are easiest and
hardest to classify. Typically, class_0 and class_2 are easier to separate
because they have more distinctive chemical profiles (e.g., high proline in
class_0, high color_intensity in class_2). Class_1, being the largest and
somewhat intermediate in feature space, may occasionally be confused with the
other two. The high F1-scores across all classes confirm that the model handles
the multiclass problem effectively.

---
## Summary

### Key Findings

1. **Data Quality:** The Wine dataset is clean with no missing values, 178 samples,
   13 continuous features, and 3 reasonably balanced classes.

2. **Feature Insights:** Features like alcohol, proline, flavanoids, and color
   intensity are the most discriminative. Several features are correlated
   (e.g., total_phenols and flavanoids), and PCA can reduce dimensionality
   while retaining most information.

3. **Model Performance:** All three models (Logistic Regression, Random Forest,
   SVM) achieve strong accuracy. Logistic Regression and SVM benefit most from
   feature scaling and typically edge out Random Forest in cross-validation.

4. **Hyperparameter Tuning:** GridSearchCV provided marginal improvements over
   default parameters, indicating the defaults are already well-suited for this
   problem. The tuning process is still valuable as a best practice.

5. **Final Model:** The tuned model achieves excellent test accuracy with strong
   precision and recall across all three wine classes.

### What Would You Try Next?

- Feature selection to simplify the model
- Ensemble methods (e.g., Gradient Boosting, stacking)
- Deeper hyperparameter search with RandomizedSearchCV
- Learning curves to assess whether more data would help
- Using PCA-reduced features with the models for comparison